In [68]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool,InjectedToolArg
from typing import Annotated 
from langchain_core.messages import HumanMessage
import requests

In [69]:
model=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",
    task='text-generation'
)

llm=ChatHuggingFace(llm=model)

In [70]:
#tool create

@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """This function fetches the currency conversion factor between a given base currency and a target currency """

    url=f'https://v6.exchangerate-api.com/v6/3b4c81b24be517898e503c78/pair/{base_currency}/{target_currency}'
    
    response=requests.get(url)

    return response.json()

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """Given a conversion rate this function calculates the target currency value from given base currency value"""

    return base_currency_value*conversion_rate


In [71]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776988801,
 'time_last_update_utc': 'Fri, 24 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1777075201,
 'time_next_update_utc': 'Sat, 25 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.1701}

In [72]:
convert.invoke({'base_currency_value':10,'conversion_rate':94.1701})

941.701

In [73]:
##tool binding

llm_with_tools=llm.bind_tools([get_conversion_factor,convert])
llm_with_tools

RunnableBinding(bound=ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='deepseek-ai/DeepSeek-V3', stop_sequences=[], server_kwargs={}, model_kwargs={}, model='deepseek-ai/DeepSeek-V3', client=<InferenceClient(model='deepseek-ai/DeepSeek-V3', timeout=120)>, async_client=<InferenceClient(model='deepseek-ai/DeepSeek-V3', timeout=120)>, task='text-generation'), model_id='deepseek-ai/DeepSeek-V3', temperature=0.8, top_p=0.95, max_tokens=512, model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetches the currency conversion factor between a given base currency and a target currency', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'convert', 'description': 'Given a conversion rate this function calculates the target currency value from given b

In [74]:
messages=[HumanMessage('what is conversion rate between USD and INR ,and based on that convert 10 usd(base_currency) to inr(target_currency)')]
messages

[HumanMessage(content='what is conversion rate between USD and INR ,and based on that convert 10 usd(base_currency) to inr(target_currency)', additional_kwargs={}, response_metadata={})]

In [75]:
ai_message=llm_with_tools.invoke(messages)
messages.append(ai_message)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_e4317e0e02624a60bc1b5763',
  'type': 'tool_call'}]

In [76]:
import json

for tool_call in ai_message.tool_calls:
    #execute 1st tool and get conversion rate
    if tool_call['name']=='get_conversion_factor':
        tool_message1=get_conversion_factor.invoke(tool_call)
        #fetch the conversion rate
        conversion_rate=json.loads(tool_message1.content)['conversion_rate']
        #append this tool message in message list
        messages.append(tool_message1)
    #execute 1st tool and get conversion rate
    if tool_call['name']=='convert':
        #fetch cureent arg
        tool_call['args']['conversion_rate']=conversion_rate
        tool_message2=convert.invoke(tool_call)
        messages.append(tool_message2)


In [77]:
messages

[HumanMessage(content='what is conversion rate between USD and INR ,and based on that convert 10 usd(base_currency) to inr(target_currency)', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'll help you convert 10 USD to INR. Let me first get the current conversion rate between USD and INR.", additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_e4317e0e02624a60bc1b5763', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 268, 'total_tokens': 319}, 'model_name': 'deepseek-ai/DeepSeek-V3', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dbe46-299e-7f01-95af-7a2aa8578ace-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_e4317e0e02624a60bc1b5763', 'type': 'tool_call'}], invalid_tool

In [78]:
llm_with_tools.invoke(messages)

AIMessage(content='Now that I have the conversion rate, let me convert 10 USD to INR using this rate:', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_82ea468dac0d4f0584146a0b', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 469, 'total_tokens': 505}, 'model_name': 'deepseek-ai/DeepSeek-V3', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dbe46-34a5-7fe2-b211-0fde72277c3f-0', tool_calls=[{'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_82ea468dac0d4f0584146a0b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 469, 'output_tokens': 36, 'total_tokens': 505})